# logit-thermo 探索 X・Y：型破りと、文の順番の戻れなさ（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 20〜40 分。
- **探索**です。確かめには使いません。生成はせず、人の文章（A2-2 と同じ BBC の 40 本）を 4 つのモデル（Pythia-1B・Qwen2.5-1.5B・OLMo-2-1B・SmolLM2-1.7B）に読ませるだけです。
- X：人の文の 1 歩ずつの「選ぶ前の迷い（エントロピー）」と「実際の一歩の意外さ（サプライザル）」を記録します。
- Y：記事の頭の数段落を、元の順番・文の逆順・段落の逆順・ばらばら（2 通り）に並べて読ませ、文章全体の確率がどれだけ下がるかを測ります。
- 中身は表示も保存もしません。保存先 `A2_explore/xy_summary.json` と `A2_explore/xy_{モデル}.npz`（数字だけ）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）と `scripts/a2_explore_xy.py`（`5c5750e76358b86a…`）をそのまま埋め込み。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 X・Y（型破りと文の順番の戻れなさ、4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 X・Y（型破りと文の順番の戻れなさ、4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore XY', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 14e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU が足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import hashlib
SRC22 = '"""A2-2（ゲート v0.3）のデータ集め。GPU のノートブックにこのファイルの中身をそのまま埋め込む。\n\n何も判定しない（分析は別の CPU ノートブック）。各モデルについて：\n- 人の文脈（teacher forcing）：各位置・各温度の ℓ（裾の取り分の log-odds）、1 位の log-odds、エントロピー。\n- 温度計の材料：β ＝ 1/T の格子ごとに Σ(⟨z⟩_β − z_y)（人のトークンの平均エネルギーとの差）。\n- a_loc・h（T ＝ 1、人の文脈の各位置）。\n- 生成：各温度・各サンプルで 512 歩、各一歩の ℓ・1 位の log-odds・エントロピーと、選んだトークン。\nすべて float64 で計算し、float32 で保存する。使わない番号（tokenizer の長さ以上）と EOS は −∞ にして外す。\n"""\nimport gc, hashlib, json, math, os, re, time\nimport numpy as np\nimport torch\n\nH = 100\nPROMPT, CONT, N_DOCS, SAMPLES = 64, 512, 40, 2\nTEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4, 1.5]\nTCAL_T = np.exp(np.linspace(np.log(0.4), np.log(2.5), 61))   # 温度計の格子（T）\nSEED = 20261027\nMODELS = [\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\', \'Pythia\'),\n    (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\', \'Qwen\'),\n    (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\', \'Qwen\'),\n    (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\', \'Qwen\'),\n    (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\', \'OLMo\'),\n    (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', \'SmolLM\'),\n    (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\', \'SmolLM\'),\n]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\')]\nBBC_MONTHS = [\'2025-04\', \'2025-05\', \'2025-06\']\nBOILER = re.compile(r\'(Getty Images|Image source|Image caption|Related topics|Follow BBC|^Watch:|^Listen:|To play this video you need to enable JavaScript)\')\nH_FRAC = 0.00066   # H を語彙の割合にした版（ゲート §2.1）\n\n\ndef h2_of(n_vocab):\n    return max(2, int(round(H_FRAC * n_vocab)))\n\n\n# ---------- 一歩の量（torch、float64） ----------\n\ndef step_stats(z, T, valid, H2):\n    """z：(B, V) の logit、valid：(V,) の bool。戻り値：ℓ（順位 H 以降）、1 位の log-odds、エントロピー、ℓ（順位 H2 以降）。\n    どれも (B,)、float64。1 位の log-odds は「1 位を除いた logsumexp」から直接計算する（p1 → 1 でも有限）。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\')) / T\n    lse = torch.logsumexp(z, dim=-1)\n    top = torch.topk(z, max(H, H2) - 1, dim=-1)\n\n    def ell_of(h):\n        head = torch.logsumexp(top.values[:, :h - 1], dim=-1)\n        return torch.logsumexp(z.scatter(-1, top.indices[:, :h - 1], float(\'-inf\')), dim=-1) - head\n    rest = torch.logsumexp(z.scatter(-1, top.indices[:, :1], float(\'-inf\')), dim=-1)\n    top1 = top.values[:, 0] - rest\n    lp = z - lse[:, None]\n    p = torch.exp(lp)\n    ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)\n    return ell_of(H), top1, ent, ell_of(H2)\n\n\ndef np_step_stats(z, T, valid, H2):\n    """自己テスト用の素直な numpy 版（1 行）。"""\n    z = np.where(valid, z.astype(np.float64), -np.inf) / T\n    zs = np.sort(z[np.isfinite(z)])[::-1]\n    def lse(x):\n        m = x.max(); return m + np.log(np.exp(x - m).sum())\n    ell = lse(zs[H - 1:]) - lse(zs[:H - 1])\n    ell2 = lse(zs[H2 - 1:]) - lse(zs[:H2 - 1])\n    lp = zs - lse(zs)\n    return ell, float(zs[0] - lse(zs[1:])), float(-(np.exp(lp) * lp).sum()), ell2\n\n\ndef shape_stats(z, valid):\n    """T ＝ 1 の a_loc（順位 300〜3,000 の傾き）と h（頭の確率 ÷ 裾の確率の log10）。z：(N, V)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zs = torch.sort(z, dim=-1, descending=True).values\n    r = torch.arange(300, 3001, device=z.device, dtype=torch.float64)\n    x = torch.log(r); x = x - x.mean()\n    y = zs[:, 299:3000]\n    a = -((y - y.mean(-1, keepdim=True)) * x).sum(-1) / (x * x).sum()\n    h = (torch.logsumexp(zs[:, :H - 1], -1) - torch.logsumexp(zs[:, H - 1:], -1)) / math.log(10)\n    return a, h\n\n\ndef tcal_terms(z, y, valid):\n    """温度計の材料：各 T（TCAL_T）で Σ_t(⟨z⟩_T − z_y)。z：(N, V)、y：(N,)。"""\n    z = z.double().masked_fill(~valid, float(\'-inf\'))\n    zy = z.gather(-1, y[:, None])[:, 0]\n    out = []\n    for T in TCAL_T:\n        lp = torch.log_softmax(z / T, dim=-1)\n        p = torch.exp(lp)\n        mean_z = torch.where(p > 0, p * z, torch.zeros_like(p)).sum(-1)\n        out.append(float((mean_z - zy).sum()))\n    return np.array(out)\n\n\n# ---------- 書き出しの文章 ----------\n\ndef clean(text):\n    paras = [p.strip() for p in text.split(\'\\n\')]\n    keep = [p for p in paras if len(p) >= 60 and not BOILER.search(p)]\n    return \'\\n\\n\'.join(keep)\n\n\ndef load_docs():\n    from datasets import load_dataset\n    pool = []\n    for m in BBC_MONTHS:\n        ds = load_dataset(\'RealTimeData/bbc_news_alltime\', m, split=\'train\')\n        for row in ds:\n            c = clean(row[\'content\'] or \'\')\n            if len(c) >= 4000:\n                pool.append({\'month\': m, \'link\': row[\'link\'], \'text\': c})\n    pool.sort(key=lambda d: hashlib.sha256(d[\'link\'].encode()).hexdigest())\n    rng = np.random.default_rng(SEED)\n    idx = rng.choice(len(pool), size=N_DOCS, replace=False)\n    docs = [pool[i] for i in sorted(idx)]\n    meta = {\'pool_size\': len(pool), \'docs\': [{\'month\': d[\'month\'], \'link_sha256\': hashlib.sha256(d[\'link\'].encode()).hexdigest(),\n                                               \'text_sha256\': hashlib.sha256(d[\'text\'].encode()).hexdigest(),\n                                               \'chars\': len(d[\'text\'])} for d in docs]}\n    return docs, meta\n\n\n# ---------- モデルごと ----------\n\ndef load_model(hf_id, device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    return tok, model\n\n\ndef valid_mask(tok, model, device):\n    V = model.get_output_embeddings().weight.shape[0]\n    valid = torch.zeros(V, dtype=torch.bool, device=device)\n    valid[:len(tok)] = True\n    eos = set()\n    if tok.eos_token_id is not None:\n        eos.add(int(tok.eos_token_id))\n    ge = getattr(model, \'generation_config\', None)\n    if ge is not None and ge.eos_token_id is not None:\n        eos |= set([ge.eos_token_id] if isinstance(ge.eos_token_id, int) else ge.eos_token_id)\n    eot = tok.convert_tokens_to_ids(\'<|endoftext|>\') if \'<|endoftext|>\' in tok.get_vocab() else None\n    if eot is not None:\n        eos.add(int(eot))   # 文書の終わりの記号も外す（SmolLM2-Instruct は EOS が <|im_end|> なので、ここで足す）\n    for e in eos:\n        if e < V:\n            valid[e] = False\n    return valid, sorted(eos)\n\n\ndef split_doc(tok, text):\n    ids = tok(text, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    if len(ids) < PROMPT + CONT:\n        return None\n    return bos + ids[:PROMPT], ids[PROMPT:PROMPT + CONT]\n\n\n@torch.no_grad()\ndef teacher_forced(model, prompt, human, valid, device, H2):\n    x = torch.tensor([prompt + human], device=device)\n    logits = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n    y = torch.tensor(human, device=device)\n    res = {}\n    for k, T in enumerate(TEMPS):\n        res[k] = [v.float().cpu().numpy() for v in step_stats(logits, T, valid, H2)]\n    a, h = shape_stats(logits, valid)\n    return res, a.float().cpu().numpy(), h.float().cpu().numpy(), tcal_terms(logits, y, valid)\n\n\n@torch.no_grad()\ndef generate(model, prompts, T, valid, gen, H2):\n    """prompts：同じ長さのトークン列のリスト。512 歩を KV キャッシュで書く。"""\n    x = torch.tensor(prompts, device=valid.device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    L, T1, E, L2, toks = [], [], [], [], []\n    for t in range(CONT):\n        ell, top1, ent, ell2 = step_stats(logits, T, valid, H2)\n        L.append(ell); T1.append(top1); E.append(ent); L2.append(ell2)\n        z = logits.double().masked_fill(~valid, float(\'-inf\')) / T\n        p = torch.softmax(z, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        toks.append(nxt[:, 0])\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda xs: torch.stack(xs, 1).float().cpu().numpy()\n    return st(L), st(T1), st(E), st(L2), torch.stack(toks, 1).int().cpu().numpy()\n\n\ndef run_model(name, hf_id, docs, outdir, device, notify=print):\n    path = f\'{outdir}/{name}.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nT, nD = len(TEMPS), len(use)\n    tf = np.zeros((4, nD, nT, CONT), np.float32)   # [ℓ(H), 1 位の log-odds, エントロピー, ℓ(H2)]\n    a_all = np.zeros((nD, CONT), np.float32); h_all = np.zeros((nD, CONT), np.float32)\n    tcal = np.zeros((nD, len(TCAL_T)))\n    for j, i in enumerate(use):\n        res, a, h, tc = teacher_forced(model, *pairs[i], valid, device, H2)\n        for k in range(nT):\n            for q in range(4):\n                tf[q, j, k] = res[k][q]\n        a_all[j], h_all[j], tcal[j] = a, h, tc\n    plen = {len(pairs[i][0]) for i in use}\n    assert len(plen) == 1, plen\n    prompts = [pairs[i][0] for i in use]\n    gen = np.zeros((4, nT, SAMPLES, nD, CONT), np.float32)\n    toks = np.zeros((nT, SAMPLES, nD, CONT), np.int32)\n    for k, T in enumerate(TEMPS):\n        for s in range(SAMPLES):\n            g = torch.Generator(device=device).manual_seed(SEED + 1000 * k + s)\n            l, t1, e, l2, tk = generate(model, prompts, T, valid, g, H2)\n            gen[0, k, s], gen[1, k, s], gen[2, k, s], gen[3, k, s], toks[k, s] = l, t1, e, l2, tk\n        print(f\'  {name} T={T} おわり（{time.time() - t0:.0f} 秒）\')\n    np.savez_compressed(path + \'.part.npz\', tf=tf, a=a_all, h=h_all, tcal=tcal, tcal_T=TCAL_T, gen=gen, toks=toks,\n                        temps=np.array(TEMPS), use=np.array(use), eos=np.array(eos), tok_len=len(tok),\n                        prompt_len=list(plen)[0], revision=str(getattr(model.config, \'_commit_hash\', \'\')),\n                        seconds=time.time() - t0, H2=H2)\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n\n\n@torch.no_grad()\ndef cross_model(reader, writer, docs, outdir, device):\n    """reader が writer の書いた文を読む。文字の窓で、人の続きを読んだときと比べる（記録用）。"""\n    path = f\'{outdir}/cross_{reader}_reads_{writer}.npz\'\n    if os.path.exists(path):\n        return\n    from transformers import AutoTokenizer\n    hf = dict((n, h) for n, h, _ in MODELS)\n    wtok = AutoTokenizer.from_pretrained(hf[writer])\n    W = np.load(f\'{outdir}/{writer}.npz\')\n    tok, model = load_model(hf[reader], device)\n    valid, _ = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    use = list(W[\'use\']); Wt = W[\'toks\']\n    out = np.full((2, len(TEMPS), SAMPLES, len(use)), np.nan)   # [cross, human] の ℓ の平均\n    for j, i in enumerate(use):\n        text = docs[i][\'text\']\n        enc = wtok(text, add_special_tokens=False, return_offsets_mapping=True)\n        pend = enc[\'offset_mapping\'][PROMPT - 1][1]                  # 書き手の書き出しの終わりの文字位置\n        pre = text[:pend]\n        for k, T in enumerate(TEMPS):\n            for s in range(SAMPLES):\n                gtxt = wtok.decode(Wt[k, s, j])\n                C = min(len(gtxt), len(text) - pend)\n                for q, cont in enumerate([gtxt[:C], text[pend:pend + C]]):\n                    e2 = tok(pre + cont, add_special_tokens=False, return_offsets_mapping=True)\n                    ids = e2[\'input_ids\']; offs = e2[\'offset_mapping\']\n                    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n                    x = torch.tensor([bos + ids], device=device)\n                    logits = model(x).logits[0].float()\n                    pos = [n for n, (a0, _) in enumerate(offs) if pend <= a0 < pend + C and n > 0]\n                    if not pos:\n                        continue\n                    pred = torch.tensor([n - 1 + len(bos) for n in pos], device=device)\n                    ell = step_stats(logits[pred], T, valid, H2)[0]\n                    out[q, k, s, j] = float(ell.mean())\n    np.savez_compressed(path + \'.part.npz\', out=out, temps=np.array(TEMPS))\n    os.replace(path + \'.part.npz\', path)\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n# ---------- 自己テスト（本番の前に） ----------\n\n@torch.no_grad()\ndef self_test(device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    H2 = 33\n    # ① 一歩の量が numpy の素直な計算と一致する（1 位が飛び抜けた行も入れる：1 位の log-odds が有限か）\n    z = torch.randn(3, valid.shape[0], device=device) * 3\n    z[2, 7] += 80.0\n    for T in [0.6, 1.0, 1.5]:\n        got_all = step_stats(z, T, valid, H2)\n        assert all(bool(torch.isfinite(v).all()) for v in got_all), (\'有限でない値\', T)\n        for b in range(3):\n            ref = np_step_stats(z[b].cpu().numpy(), T, valid.cpu().numpy(), H2)\n            got = tuple(float(v[b]) for v in got_all)\n            assert all(abs(g - r) < 1e-6 * max(1, abs(r)) for g, r in zip(got, ref)), (T, got, ref)\n    # ② KV キャッシュの生成と、まとめて計算した logit が一致する\n    g = torch.Generator(device=device).manual_seed(0)\n    prompts = [list(range(5, 5 + 16)), list(range(30, 30 + 16))]\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    seq, steps = x, []\n    for t in range(8):\n        nxt = torch.multinomial(torch.softmax(lg.masked_fill(~valid, float(\'-inf\')), -1), 1, generator=g)\n        steps.append(lg.clone()); seq = torch.cat([seq, nxt], 1)\n        out = model(nxt, past_key_values=past, use_cache=True); past, lg = out.past_key_values, out.logits[:, -1, :]\n    full = model(seq).logits\n    for t in range(8):\n        assert torch.allclose(full[:, 15 + t, :], steps[t], atol=1e-4), (\'KV キャッシュの不一致\', t)\n    # ③ 1 歩目は生成と人の文脈で同じ分布\n    human = list(range(100, 100 + 64))   # 小さなモデルは位置 512 までなので短くする（使うのは位置 15 だけ）\n    tf_logits = model(torch.tensor([prompts[0] + human], device=device)).logits[0, 15]\n    gen_first = model(torch.tensor([prompts[0]], device=device)).logits[0, -1]\n    assert torch.allclose(tf_logits, gen_first, atol=1e-5), \'1 歩目が一致しない\'\n    return \'OK\'\n'
SRCXY = '"""探索 X（型破り）と Y（文の順番の戻れなさ）の下見。2026-10-28 夜。確かめには使わない。\nGPU（L4）のノートブックで、scripts/a2_2_collect.py を実行したのと同じ名前空間で実行する\n（load_docs・split_doc・load_model・valid_mask・step_stats・h2_of・PROMPT・CONT を使う）。生成はしない（人の文章を読むだけ）。\n\nモデル：A2-2 の素のモデル 4 つ（Pythia-1B、Qwen2.5-1.5B、OLMo-2-1B、SmolLM2-1.7B）。記事：A2-2 と同じ BBC の 40 本。\n\nX（型破り）：A2-2 と同じ書き出し 64 ＋ 人の続き 512 トークンで、温度 1 の分布について、各歩の\n  エントロピー H（選ぶ前の迷い、ナット）、サプライザル s ＝ −log p（実際の一歩の意外さ、ナット）、ℓ（H ＝ 100）、1 位の log-odds、\n  人のトークンの順位、を記録する（xy_{モデル}.npz に保存）。\n  読み方（回す前に決める。探索の目安）：\n  - 型破り歩：H < 1.0 かつ s > H ＋ 3.0（迷いは小さいのに、意外さが期待値より 3 ナット以上大きい）。\n  - 分かれ道の歩：H > 3.0。\n  - 記録する量：型破り歩の割合、型破り歩が NLL（s の合計）に占める割合、分かれ道の歩の割合と NLL に占める割合、\n    位置の帯（1〜8・8〜32・32〜128・128〜512）ごとの型破り歩の割合。\n\nY（文の順番の戻れなさ）：記事の頭から段落を足していき、合わせて 2,500 字を超えないところまでを「窓」にする\n  （少なくとも 2 段落・6 文）。窓の文を、次の並びにして、固定の書き出し "Article:\\\\n" のあとに置いた文章の\n  対数確率（書き出しのあとの全トークンの log p の合計、温度 1、外した番号は除く）を測る。文はすべて半角スペース 1 つでつなぐ。\n  書き出しと本文は別々にトークンに区切ってからつなぐ。\n  - original：元の順番。\n  - sent_rev：文の順番を逆にする。\n  - para_rev：段落の順番を逆にする（段落の中の文の順番はそのまま）。\n  - shuf1・shuf2：文の順番をばらばらにする（種 1・2）。\n  - 戻れなさ σ ＝ log p（original）− log p（並べ替え）。記事ごとの値（ナット）と、トークンあたりの値。\n  読み方（回す前に決める。探索の目安）：\n  - 検量線：sent_rev と shuf で σ > 0 の記事が 9 割以上なら「ものさしは使える」。そうでなければ、この探索の言葉は使わない。\n  - 粗く見ると小さくなるか：para_rev の σ が sent_rev の σ 以下の記事の割合（確率的な熱力学では、粗く見たときの\n    エントロピー生成は下限になる。ここでは「段落の逆順」を「文の逆順」より粗い並べ替えとみなす）。\n  - 逆順とばらばらの比べ：σ(sent_rev) と σ(shuf) の平均の比（逆順は、ばらばらより「戻れない」か）。\n  文の分け方：段落の中を、. ! ? のあとの空白で分ける（20 字未満の断片は前の文につなぐ）。中身は表示しない・保存しない。\n"""\nimport gc, json, math, re, time\nimport numpy as np\nimport torch\n\nXY_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'),\n             (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\')]\nX_LOW_H, X_JUMP, FORK_H = 1.0, 3.0, 3.0\nY_CHARS, Y_MIN_PARA, Y_MIN_SENT = 2500, 2, 6\nY_PREFIX = \'Article:\\n\'\nX_BINS = [(1, 8), (8, 32), (32, 128), (128, 512)]\n\n\ndef split_sentences(para):\n    parts = [p.strip() for p in re.split(r\'(?<=[.!?])\\s+\', para) if p.strip()]\n    out = []\n    for p in parts:\n        if out and len(p) < 20:\n            out[-1] = out[-1] + \' \' + p\n        else:\n            out.append(p)\n    return out\n\n\ndef y_window(text):\n    """→ 段落ごとの文のリスト（list of list）。条件を満たさなければ None。"""\n    paras = [p for p in text.split(\'\\n\\n\') if p.strip()]\n    win, n = [], 0\n    for p in paras:\n        if n + len(p) > Y_CHARS and len(win) >= Y_MIN_PARA:\n            break\n        win.append(split_sentences(p)); n += len(p)\n    if len(win) < Y_MIN_PARA or sum(len(s) for s in win) < Y_MIN_SENT:\n        return None\n    return win\n\n\ndef y_orders(win):\n    sents = [s for p in win for s in p]\n    out = {\'original\': sents, \'sent_rev\': sents[::-1], \'para_rev\': [s for p in win[::-1] for s in p]}\n    for k in [1, 2]:\n        r = np.random.default_rng(k).permutation(len(sents))\n        out[f\'shuf{k}\'] = [sents[i] for i in r]\n    return {k: \' \'.join(v) for k, v in out.items()}\n\n\n@torch.no_grad()\ndef seq_logprob(model, tok, text, valid, device):\n    pre = tok(Y_PREFIX, add_special_tokens=False).input_ids\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    ids = pre + tok(text, add_special_tokens=False).input_ids      # 書き出しと本文は別々に区切ってつなぐ（境目で区切りが変わらないように）\n    x = torch.tensor([bos + ids], device=device)\n    z = model(x).logits[0].double().masked_fill(~valid, float(\'-inf\'))\n    lp = torch.log_softmax(z, -1)\n    start = len(bos) + len(pre)\n    y = x[0, start:]\n    tgt = lp[start - 1:x.shape[1] - 1].gather(-1, y[:, None])[:, 0]\n    return float(tgt.sum()), int(len(y))\n\n\n@torch.no_grad()\ndef x_steps(model, prompt, human, valid, device, H2):\n    x = torch.tensor([prompt + human], device=device)\n    logits = model(x).logits[0, len(prompt) - 1: len(prompt) - 1 + CONT].float()\n    z = logits.double().masked_fill(~valid, float(\'-inf\'))\n    lp = torch.log_softmax(z, -1)\n    y = torch.tensor(human, device=device)\n    s = -lp.gather(-1, y[:, None])[:, 0]\n    p = torch.exp(lp)\n    Hh = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)\n    ell, top1, _, _ = step_stats(logits, 1.0, valid, H2)\n    rank = (z > z.gather(-1, y[:, None])).sum(-1)\n    f = lambda v: v.float().cpu().numpy()\n    return dict(H=f(Hh), s=f(s), ell=f(ell), top1=f(top1), rank=rank.int().cpu().numpy())\n\n\ndef x_summary(X):\n    H, s = X[\'H\'].astype(np.float64), X[\'s\'].astype(np.float64)      # (nD, L)\n    jump = (H < X_LOW_H) & (s > H + X_JUMP)\n    fork = H > FORK_H\n    tot = s.sum()\n    return dict(jump_rate=float(jump.mean()), jump_share_nll=float(s[jump].sum() / tot),\n                fork_rate=float(fork.mean()), fork_share_nll=float(s[fork].sum() / tot),\n                jump_rate_by_bin={f\'{a}-{b}\': float(jump[:, a:b].mean()) for a, b in X_BINS},\n                nll_per_token=float(s.mean()), mean_H=float(H.mean()))\n\n\ndef y_summary(Y):\n    """Y：記事ごとの {order: (logp, ntok)} のリスト。"""\n    out = {}\n    lp = {k: np.array([d[k][0] for d in Y]) for k in Y[0]}\n    nt = {k: np.array([d[k][1] for d in Y]) for k in Y[0]}\n    for k in [\'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']:\n        sig = lp[\'original\'] - lp[k]\n        out[k] = dict(sigma_mean=float(sig.mean()), sigma_median=float(np.median(sig)),\n                      sigma_per_token_mean=float((sig / nt[\'original\']).mean()), frac_positive=float((sig > 0).mean()),\n                      ntok_diff_mean=float((nt[k] - nt[\'original\']).mean()))\n    shuf_frac = min(out[\'shuf1\'][\'frac_positive\'], out[\'shuf2\'][\'frac_positive\'])\n    out[\'calibration_ok\'] = bool(out[\'sent_rev\'][\'frac_positive\'] >= 0.9 and shuf_frac >= 0.9)\n    s_sent, s_para = lp[\'original\'] - lp[\'sent_rev\'], lp[\'original\'] - lp[\'para_rev\']\n    out[\'frac_para_le_sent\'] = float((s_para <= s_sent).mean())\n    s_shuf = ((lp[\'original\'] - lp[\'shuf1\']) + (lp[\'original\'] - lp[\'shuf2\'])) / 2\n    out[\'ratio_rev_over_shuf\'] = float(s_sent.mean() / s_shuf.mean()) if s_shuf.mean() != 0 else float(\'nan\')\n    out[\'n_docs\'] = len(Y)\n    return out\n\n\ndef run_xy(docs, outdir, device, notify=print):\n    res = {}\n    wins = [y_window(d[\'text\']) for d in docs]\n    res[\'y_docs_used\'] = int(sum(w is not None for w in wins))\n    for name, hf_id in XY_MODELS:\n        path = f\'{outdir}/xy_{name}.npz\'\n        t0 = time.time()\n        tok, model = load_model(hf_id, device)\n        valid, _ = valid_mask(tok, model, device)\n        H2 = h2_of(len(tok))\n        pairs = [split_doc(tok, d[\'text\']) for d in docs]\n        use = [i for i, p in enumerate(pairs) if p is not None]\n        Xs = [x_steps(model, *pairs[i], valid, device, H2) for i in use]\n        X = {k: np.stack([x[k] for x in Xs]) for k in Xs[0]}\n        Y = []\n        for w in wins:\n            if w is None:\n                continue\n            Y.append({k: seq_logprob(model, tok, t, valid, device) for k, t in y_orders(w).items()})\n        np.savez_compressed(path, use=np.array(use), **X,\n                            y_logp=np.array([[d[k][0] for k in [\'original\', \'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']] for d in Y]),\n                            y_ntok=np.array([[d[k][1] for k in [\'original\', \'sent_rev\', \'para_rev\', \'shuf1\', \'shuf2\']] for d in Y]))\n        res[name] = dict(x=x_summary(X), y=y_summary(Y), seconds=round(time.time() - t0, 1))\n        del model; gc.collect(); torch.cuda.empty_cache()\n        notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n    return res\n\n\n@torch.no_grad()\ndef self_test_xy(device):\n    """小さなランダムのモデルで：文の分け方と並べ替え、seq_logprob と x_steps が素直な計算と合うか。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    # 文の分け方・窓・並べ替え\n    para = \'One sentence here. Two is here! Three? ok. Four is long enough here.\'\n    ss = split_sentences(para)\n    assert ss[-1].endswith(\'here.\') and all(len(s) >= 9 for s in ss), ss\n    text = \'\\n\\n\'.join([\'Alpha one is a sentence. Alpha two is another one. Alpha three ends here.\'] * 2 +\n                       [\'Beta one is a sentence. Beta two is another one.\'])\n    w = y_window(text)\n    assert w is not None and len(w) == 3\n    od = y_orders(w)\n    assert sorted(od[\'original\'].split()) == sorted(od[\'sent_rev\'].split()) == sorted(od[\'shuf1\'].split())\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    lp, n = seq_logprob(model, tok, od[\'original\'], valid, device)\n    pre_ids = tok(Y_PREFIX, add_special_tokens=False).input_ids\n    ids = pre_ids + tok(od[\'original\'], add_special_tokens=False).input_ids\n    z = model(torch.tensor([ids], device=device)).logits[0].double().masked_fill(~valid, float(\'-inf\'))\n    ls = torch.log_softmax(z, -1)\n    pre = len(pre_ids)\n    ref = sum(float(ls[i - 1, ids[i]]) for i in range(pre, len(ids)))\n    assert abs(lp - ref) < 1e-6 * max(1, abs(ref)) and n == len(ids) - pre, (lp, ref)\n    global CONT\n    keep = CONT\n    CONT = 20\n    try:\n        prompt, human = list(range(5, 13)), list(range(20, 40))\n        X = x_steps(model, prompt, human, valid, device, 33)\n        full = model(torch.tensor([prompt + human], device=device)).logits[0].double().masked_fill(~valid, float(\'-inf\'))\n        ls = torch.log_softmax(full, -1)\n        for t in [0, 7, 19]:\n            assert abs(float(X[\'s\'][t]) + float(ls[len(prompt) - 1 + t, human[t]])) < 1e-4\n        assert np.all(X[\'H\'] >= 0) and np.all(X[\'s\'] >= 0)\n        sm = x_summary({k: v[None] for k, v in X.items()})\n        assert 0 <= sm[\'jump_rate\'] <= 1\n    finally:\n        CONT = keep\n    return \'OK\'\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRCXY.encode()).hexdigest() == '5c5750e76358b86a1840e6894f9cd86c377e6be1f0b9d4104998b5144a1faaea'
exec(SRC22)
exec(SRCXY)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 モデルで X と Y → 保存（結果は表示しない）
import json
try:
    print('自己テスト：', self_test('cuda'), self_test_xy('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore', exist_ok=True)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = run_xy(docs, '/content/drive/MyDrive/logit-thermo/A2_explore', 'cuda', notify=lambda m: notify('✅ ' + m, tags='white_check_mark'))
    r['a2_explore_xy_sha256'] = '5c5750e76358b86a1840e6894f9cd86c377e6be1f0b9d4104998b5144a1faaea'
    json.dump(r, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/xy_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 X・Yで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：型破りと文の順番の戻れなさを A2_explore に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
